# DE-05 — Orchestration & Reliability

**Dataset:** `data/loan_data_05.csv`

This notebook demonstrates DAG dependencies, schedules, event triggers, bounded retries with backoff, timeout metadata, concurrency and resource pools, idempotency, checkpoints, atomic publication, backfill, SLA/SLO, failure routing, and a small operational runbook.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_05.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

Dataset: loan_data_05.csv
Rows: 48 | Columns: 13
 Loan_ID Gender Married Dependents Education Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History Property_Area Loan_Status
LP001977   Male     Yes          1  Graduate            No             1625             1803.0        96.0             360.0             1.0         Urban           Y
LP001978   Male      No          0  Graduate            No             4000             2500.0       140.0             360.0             1.0         Rural           Y
LP001993 Female      No          0  Graduate            No             3762             1666.0       135.0             360.0             1.0         Rural           Y


## Learning Content

- A DAG makes dependencies explicit and prevents publication before prerequisites succeed.
- Schedules and event triggers describe *when* a run is eligible.
- Retries must be bounded, delayed, and safe.
- Timeouts limit waiting; concurrency and resource pools protect shared systems.
- Idempotency ensures a repeated run reaches the same state.
- Checkpoints record committed progress.
- Atomic publish hides partial output.
- Backfill is a bounded historical replay.
- SLA/SLO define measurable expectations.
- Failure routing and runbooks connect technical signals to human action.

In [2]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from time import perf_counter
from typing import Callable
import uuid

@dataclass
class DemoTask:
    name: str
    action: Callable[[], object]
    upstream: tuple[str, ...] = ()
    max_attempts: int = 1
    backoff_seconds: float = 0.0
    timeout_seconds: float = 30.0
    resource_pool: str = "default"

@dataclass
class DemoDAG:
    name: str
    schedule: str
    event_triggers: set[str]
    max_concurrency: int
    resource_pools: dict[str, int]
    tasks: dict[str, DemoTask] = field(default_factory=dict)

    def add(self, task: DemoTask) -> None:
        if task.name in self.tasks:
            raise ValueError(f"Duplicate task: {task.name}")
        self.tasks[task.name] = task

    def ordered_tasks(self):
        pending = dict(self.tasks)
        completed = set()
        while pending:
            ready = sorted(
                [task for task in pending.values()
                 if set(task.upstream).issubset(completed)],
                key=lambda task: task.name,
            )
            if not ready:
                raise ValueError("Unknown dependency or cycle")
            for task in ready:
                yield task
                completed.add(task.name)
                pending.pop(task.name)

## Hands-on / Demonstration

### Define a DAG and inject a controlled failure

The target is a dictionary keyed by `Loan_ID`, making replay idempotent. Candidate output is prepared separately and published only after validation.

In [3]:
attempts = {"load_transform": 0}
checkpoint = {}
published_store = {}
failure_route = []
run_id = str(uuid.uuid4())

def source_ready():
    assert DATA_FILE.exists()
    return {"source": DATA_FILE.name, "rows": len(raw)}

def load_transform():
    attempts["load_transform"] += 1
    if attempts["load_transform"] == 1:
        raise RuntimeError("Injected training failure")
    candidate = raw.copy()
    candidate["TotalIncome"] = candidate["ApplicantIncome"] + candidate["CoapplicantIncome"]
    return candidate.drop_duplicates("Loan_ID", keep="last")

def validate_candidate(candidate):
    assert candidate["Loan_ID"].is_unique
    assert candidate["Loan_Status"].isin(["Y", "N"]).all()
    return candidate

def atomic_publish(candidate):
    # Build a complete replacement first, then swap the reference.
    replacement = candidate.set_index("Loan_ID").to_dict("index")
    published_store.clear()
    published_store.update(replacement)
    checkpoint["run_id"] = run_id
    checkpoint["published_at"] = datetime.now(timezone.utc).isoformat()
    checkpoint["row_count"] = len(replacement)
    return len(replacement)

candidate_holder = {}

dag = DemoDAG(
    name="loan_training_pipeline",
    schedule="0 2 * * *",
    event_triggers={"source_updated", "backfill_requested"},
    max_concurrency=2,
    resource_pools={"source": 1, "warehouse": 1},
)
dag.add(DemoTask("source_ready", source_ready, resource_pool="source"))
dag.add(DemoTask(
    "load_transform",
    lambda: candidate_holder.setdefault("data", load_transform()),
    upstream=("source_ready",),
    max_attempts=3,
    backoff_seconds=0.01,
    timeout_seconds=30,
    resource_pool="warehouse",
))
dag.add(DemoTask(
    "validate",
    lambda: validate_candidate(candidate_holder["data"]),
    upstream=("load_transform",),
    resource_pool="warehouse",
))
dag.add(DemoTask(
    "publish",
    lambda: atomic_publish(candidate_holder["data"]),
    upstream=("validate",),
    resource_pool="warehouse",
))

In [4]:
def run_with_retries(dag: DemoDAG, trigger: str = "manual"):
    if trigger != "manual" and trigger not in dag.event_triggers:
        raise ValueError(f"Unsupported trigger: {trigger}")
    results = {}
    started = perf_counter()

    for task in dag.ordered_tasks():
        for attempt in range(1, task.max_attempts + 1):
            try:
                results[task.name] = task.action()
                break
            except Exception as error:
                failure_route.append({
                    "task": task.name,
                    "attempt": attempt,
                    "error": str(error),
                    "owner": "Data Engineering",
                })
                if attempt == task.max_attempts:
                    raise
                # Training uses a tiny delay; production uses real exponential backoff.
                import time
                time.sleep(task.backoff_seconds * (2 ** (attempt - 1)))

    return results, perf_counter() - started

results, duration_seconds = run_with_retries(dag)
print("Task results:", {k: type(v).__name__ for k, v in results.items()})
print("Attempts:", attempts)
print("Failure route:", failure_route)
print("Published rows:", len(published_store))

Task results: {'source_ready': 'dict', 'load_transform': 'DataFrame', 'validate': 'DataFrame', 'publish': 'int'}
Attempts: {'load_transform': 2}
Failure route: [{'task': 'load_transform', 'attempt': 1, 'error': 'Injected training failure', 'owner': 'Data Engineering'}]
Published rows: 48


### Idempotent rerun, checkpoint, and bounded backfill

In [5]:
# Rerun publication with the same source.
# DataFrame comparison treats missing values in matching positions as equal.
first_state = pd.DataFrame.from_dict(published_store, orient="index").sort_index()
candidate_holder.clear()
attempts["load_transform"] = 1  # next call succeeds directly
rerun_results, _ = run_with_retries(dag)
rerun_state = pd.DataFrame.from_dict(published_store, orient="index").sort_index()
pd.testing.assert_frame_equal(first_state, rerun_state)

# The dataset has no event date, so this bounded backfill uses row positions
# as a training partition range.
backfill_start, backfill_end = 10, 20
bounded_backfill = raw.iloc[backfill_start:backfill_end].copy()

assert 0 <= backfill_start < backfill_end <= len(raw)
assert checkpoint["row_count"] == len(raw)
print("Checkpoint:", checkpoint)
print("Bounded backfill rows:", len(bounded_backfill))
print("Idempotent rerun verified.")

Checkpoint: {'run_id': '7a799f5d-53a5-45b6-9ff6-4fdd6a83abb0', 'published_at': '2026-08-26T07:10:08.199737+00:00', 'row_count': 48}
Bounded backfill rows: 10
Idempotent rerun verified.


### SLA/SLO, failure routing, and operational runbook

In [6]:
sla_seconds = 60
slo_success_target = 0.99
sla_met = duration_seconds <= sla_seconds

runbook = {
    "alert": "loan pipeline failed or exceeded SLA",
    "owner": "Data Engineering",
    "first_checks": [
        "Confirm source availability and schema",
        "Inspect failed task and last committed checkpoint",
        "Measure warehouse health and lock contention",
    ],
    "safe_recovery": "Fix the cause, then replay the bounded idempotent run",
    "escalation": "Notify Loan Analytics owner if serving freshness is at risk",
}

assert sla_met
assert failure_route[0]["task"] == "load_transform"
print(f"Duration: {duration_seconds:.4f}s | SLA: {sla_seconds}s | Met: {sla_met}")
print("SLO success target:", slo_success_target)
print("Runbook:", runbook)

Duration: 0.0273s | SLA: 60s | Met: True
SLO success target: 0.99
Runbook: {'alert': 'loan pipeline failed or exceeded SLA', 'owner': 'Data Engineering', 'first_checks': ['Confirm source availability and schema', 'Inspect failed task and last committed checkpoint', 'Measure warehouse health and lock contention'], 'safe_recovery': 'Fix the cause, then replay the bounded idempotent run', 'escalation': 'Notify Loan Analytics owner if serving freshness is at risk'}


## Enterprise Control

Retries must be bounded and safe; repeated writes must not corrupt or duplicate data.

Concurrency must also be deliberate: independent tasks may run concurrently, but database and source resource pools must cap pressure. Timeout implementations must terminate or isolate underlying work—not merely stop waiting for a thread.

In [7]:
assert all(task.max_attempts >= 1 for task in dag.tasks.values())
assert dag.resource_pools["warehouse"] == 1
assert len(published_store) == len(set(published_store))
pd.testing.assert_frame_equal(first_state, rerun_state)
assert checkpoint["row_count"] == len(raw)

print("DE-05 controls passed.")

DE-05 controls passed.


## PostgreSQL Execution

Run this cell after the learning and hands-on sections. It executes the same PostgreSQL pipeline used by the Python scripts, using this notebook's matched CSV partition and the active .env configuration.

The published results are available in `control.pipeline_runs`, `bronze`, `silver`, and `gold`.

In [8]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from retailion.pipeline import run as run_postgres_pipeline

# This executes the same PostgreSQL pipeline as scripts/run_pipeline.py.
# The matched CSV partition for this notebook is used as the source.
run_postgres_pipeline(DATA_FILE, load_mode="full")
print(f"PostgreSQL pipeline completed for {DATA_FILE.name}.")


PostgreSQL pipeline completed for loan_data_05.csv.
